In [18]:
import numpy as np
import pandas as pd

from abs_affinity_based_slotting.config import RAW_DIR, PROCESSED_DIR, DOCK, inches_to_meters

from abs_affinity_based_slotting.data.loaders import WarehouseDataLoader
from abs_affinity_based_slotting.data.split import split_picking_events
from abs_affinity_based_slotting.data.io import read_parquet
from abs_affinity_based_slotting.data.schemas import validate_dataset_tables

from abs_affinity_based_slotting.demand import build_sku_demand
from abs_affinity_based_slotting.warehouse import (
    build_bay_distance_matrix,
    distance_to_dock,
    build_locations,
    occupied_locations,
    build_location_costs,
)


from abs_affinity_based_slotting.slotting import Assignment




pd.set_option("display.width", 150)


In [2]:
data = WarehouseDataLoader(RAW_DIR).load_all()
for name in ("coordinates", "distances", "initial_stock", "picking_events", "replenishment_events"):
    df = getattr(data, name)
    print(f"{name:24s} {df.shape}")

coordinates              (1001, 7)
distances                (500500, 3)
initial_stock            (30000, 6)
picking_events           (174597, 9)
replenishment_events     (14647, 8)


In [3]:
validate_dataset_tables(
    coordinates=data.coordinates,
    distances=data.distances,
    initial_stock=data.initial_stock,
    picking_events=data.picking_events,
    replenishment_events=data.replenishment_events,
)


In [4]:
split = split_picking_events(data.picking_events, test_size=0.2)


In [5]:
D = build_bay_distance_matrix(data.distances)

locations = build_locations(data.initial_stock)
occupied_locations = occupied_locations(data.initial_stock)
costs = build_location_costs(data.initial_stock, data.distances)



In [6]:
# un batch esta en train o en test, no en ambos
assert set(split.train.batch_id) & set(split.test.batch_id) == set(), "fuga de batches!"


In [7]:
# orden temporal por INICIO de batch
last_train_start = split.train.groupby("batch_id").timestamp.min().max()
first_test_start = split.test.groupby("batch_id").timestamp.min().min()
assert last_train_start < first_test_start, "solapamiento de inicio de batch!"

In [8]:
print(f"cutoff:        {split.cutoff}")
print(f"train lines:   {len(split.train):>7,} | batches: {split.train.batch_id.nunique()}")
print(f"test  lines:   {len(split.test):>7,} | batches: {split.test.batch_id.nunique()}")

cutoff:        2025-01-25 16:00:51.933333
train lines:   139,632 | batches: 1600
test  lines:    34,965 | batches: 400


In [9]:
sku_demand = build_sku_demand(split.train)
sku_demand.head()

,sku,merchant_account_id,pick_lines,total_units,unique_batches,first_pick_date,last_pick_date
0,SKU-09522,MER-001,2208,4377,1199,2025-01-01 08:10:45.333333,2025-01-25 15:17:07.733333
1,SKU-00169,MER-009,2205,4335,1230,2025-01-01 08:00:27.533333,2025-01-25 16:05:06.933333
2,SKU-19881,MER-008,2140,4260,1173,2025-01-01 08:48:17.933333,2025-01-25 16:12:44.733333
3,SKU-03808,MER-003,2127,4300,1174,2025-01-01 08:04:55.933333,2025-01-25 16:01:26.533333
4,SKU-04654,MER-010,2111,4212,1206,2025-01-01 08:05:37.533333,2025-01-25 15:14:43.533333


In [10]:
# pick_lines >= unique_batches (puede haber varias líneas del mismo SKU por batch)
assert (sku_demand.pick_lines >= sku_demand.unique_batches).all()

### matriz de distancia

In [11]:
D = build_bay_distance_matrix(data.distances)
print(D.shape)
D.head()

(1001, 1001)


bay_id,A01-01,A01-02,A01-03,A01-04,A01-05,A01-06,A01-07,A01-08,A01-09,A01-10,...,A25-32,A25-33,A25-34,A25-35,A25-36,A25-37,A25-38,A25-39,A25-40,DOCK
bay_id,,,,,,,,,,,,,,,,,,,,,
A01-01,0.0,0.0,96.0,96.0,192.0,192.0,288.0,288.0,384.0,384.0,...,6168.0,6264.0,6264.0,6360.0,6360.0,6456.0,6456.0,6552.0,6552.0,464.0
A01-02,0.0,0.0,96.0,96.0,192.0,192.0,288.0,288.0,384.0,384.0,...,6168.0,6264.0,6264.0,6360.0,6360.0,6456.0,6456.0,6552.0,6552.0,464.0
A01-03,96.0,96.0,0.0,0.0,96.0,96.0,192.0,192.0,288.0,288.0,...,6072.0,6168.0,6168.0,6264.0,6264.0,6360.0,6360.0,6456.0,6456.0,560.0
A01-04,96.0,96.0,0.0,0.0,96.0,96.0,192.0,192.0,288.0,288.0,...,6072.0,6168.0,6168.0,6264.0,6264.0,6360.0,6360.0,6456.0,6456.0,560.0
A01-05,192.0,192.0,96.0,96.0,0.0,0.0,96.0,96.0,192.0,192.0,...,5976.0,6072.0,6072.0,6168.0,6168.0,6264.0,6264.0,6360.0,6360.0,656.0


In [12]:
assert D.shape[0] == D.shape[1], "matriz no cuadrada"
assert np.allclose(D.values, D.values.T, equal_nan=True), "matriz no simétrica"
assert (np.diag(D.values) == 0).all(), "diagonal no nula"

In [13]:
# chek contra una fila "cruda"
row = data.distances.iloc[42]
assert D.loc[row["bay_a"], row["bay_b"]] == row["distance_in"]

In [14]:
dock_dist = distance_to_dock(data.distances)
assert dock_dist.loc[DOCK] == 0.0
assert dock_dist.drop(DOCK).gt(0).all(), "hay bays a distancia 0 del dock"

### Costos de acceso

In [15]:
locations = build_locations(data.initial_stock)
locations.head()

,location_id,location_name,bay_id,sku,merchant_account_id,units,is_empty
0,1,A01-01-A-01,A01-01,SKU-00001,MER-001,12,False
1,2,A01-01-A-02,A01-01,SKU-00002,MER-010,19,False
2,3,A01-01-A-03,A01-01,SKU-00003,MER-001,10,False
3,4,A01-01-A-04,A01-01,SKU-00004,MER-009,16,False
4,5,A01-01-A-05,A01-01,SKU-00005,MER-007,43,False


In [16]:
occupied = occupied_locations(data.initial_stock)
occupied.head()

TypeError: 'DataFrame' object is not callable

In [ ]:
assert locations["is_empty"].sum() + len(occupied) == len(locations)

In [ ]:
print(f"locations: {len(locations):,} | ocupadas: {len(occupied):,} | vacías: {locations.is_empty.sum():,}")

In [ ]:
costs = build_location_costs(data.initial_stock, data.distances)
costs.head()

In [ ]:

assert costs.distance_to_dock_in.notna().all(), "costos con NaN"
assert np.allclose(costs.distance_to_dock_m, inches_to_meters(costs.distance_to_dock_in))
print(f"location_costs: {len(costs):,} filas | "
      f"dock {costs.distance_to_dock_m.min():.1f}")


In [ ]:
expected = ["picking_train", "picking_test", "sku_demand", "location_costs"]
processed = {name: read_parquet(PROCESSED_DIR / f"{name}.parquet") for name in expected}

for name, df in processed.items():
    print(f"{name:16s} {df.shape}")

# coherencia: lo recién calculado coincide con lo guardado
assert len(processed["picking_train"]) == len(split.train)
assert len(processed["sku_demand"]) == len(sku_demand)
assert len(processed["location_costs"]) == len(costs)
print("\nartefactos processed coherentes con el cálculo en vivo — todo OK")

### Slotting / Assignments

In [19]:
a = Assignment({"SKU-1": 10, "SKU-2": 20, "SKU-3": 30})

In [20]:
# busqueda en ambas direcciones
assert a.location_of("SKU-2") == 20
assert a.sku_at(30) == "SKU-3"
assert a.sku_at(99) is None
assert len(a) == 3 and "SKU-1" in a

In [21]:
# el swap funciona
a.swap("SKU-1", "SKU-3")
assert a.location_of("SKU-1") == 30 and a.sku_at(10) == "SKU-3"
a.swap("SKU-1", "SKU-3")
assert a.location_of("SKU-1") == 10

In [22]:
# dos SKUs en la misma location -> error
try:
    Assignment({"SKU-1": 5, "SKU-2": 5})
    raise AssertionError("debió rechazar location duplicada")
except ValueError:
    pass

In [26]:
# desde un dataframe
import pandas as pd
df = pd.DataFrame({"sku": ["A", "B"], "location_id": [1, 2]})
assert Assignment.from_frame(df).to_frame().equals(df) # como que los dataframe no igualan con ==

<class 'pandas.DataFrame'>
